# Montreal Newcomer Navigator — Score Calculator

Computes 5 columns from `Master_Neighborhoods`:

- **Transit_Score_0_100**
- **Campus_Access_Score_0_100**
- **Affordability_Score_0_100**
- **Amenity_Score_0_100**
- **Current_Usability_Score_0_100** (overall composite)

All component scores are **min-max normalized to 0–100** across the 34 neighborhoods (best = ~100, worst = ~0), then combined with weights explained in each section below.

In [1]:
import pandas as pd
import numpy as np

INPUT_XLSX = "Montreal_Newcomer_Navigator_Master_Dataset_updated.xlsx"
OUTPUT_CSV = "Montreal_Newcomer_Navigator_Scored.csv"

df = pd.read_excel(INPUT_XLSX, sheet_name="Master_Neighborhoods")
df.head()

,Neighborhood,Boundary_Type,CodeID,Centroid_Lat,Centroid_Lon,Area_sq_km,Environment_Type,Transit_Stop_Count,Metro_Station_Count,Accessible_Stop_Count,...,Library_Count,Park_Count,Gym_Count,Amenity_Data_Status,Sample_Transit_Stops,Transit_Score_0_100,Campus_Access_Score_0_100,Affordability_Score_0_100,Amenity_Score_0_100,Current_Usability_Score_0_100
0,Montréal-Ouest,Ville liée,30,45.452764,-73.649078,1.42,Mixed urban/suburban,21,0,19,...,1,1,1,Estimated via population/density modeling + we...,"Westminster / Westover, Westminster / Garden, ...",100.0,98.9,Pending neighborhood rent data; CMA rent value...,Pending amenity import,99.5
1,Westmount,Ville liée,22,45.485348,-73.600724,4.02,Urban,82,0,81,...,1,6,10,Estimated via population/density modeling + we...,"The Boulevard / Victoria, René-Lévesque / Atwa...",76.6,90.7,Pending neighborhood rent data; CMA rent value...,Pending amenity import,82.9
2,Ville-Marie,Arrondissement,13,45.509345,-73.555543,21.50,Urban,501,16,430,...,4,31,74,Estimated via population/density modeling + we...,"STATION ATWATER, Station Atwater, Station Atwa...",62.3,100.0,Pending neighborhood rent data; CMA rent value...,Pending amenity import,79.3
3,Le Plateau-Mont-Royal,Arrondissement,15,45.526103,-73.580750,8.14,Urban,355,3,338,...,4,33,53,Estimated via population/density modeling + we...,"STATION SHERBROOKE, Station Sherbrooke, Statio...",61.7,91.7,Pending neighborhood rent data; CMA rent value...,Pending amenity import,75.2
4,Côte-des-Neiges-Notre-Dame-de-Grâce,Arrondissement,20,45.484885,-73.631757,21.49,Mixed urban/suburban,658,9,619,...,5,43,48,Estimated via population/density modeling + we...,"STATION DE LA SAVANE, Station De la Savane, St...",45.1,89.4,Pending neighborhood rent data; CMA rent value...,Pending amenity import,65.0


## Helper: min-max normalization

Scales any numeric column to 0–100. `invert=True` flips it so a *lower* raw value (e.g. distance, rent) produces a *higher* score. If every neighborhood has the same value (no spread), everyone gets a neutral 50 rather than a divide-by-zero error.

In [2]:
def minmax(series, invert=False):
    """Scale a numeric series to 0-100. invert=True means lower raw value -> higher score."""
    s = series.astype(float)
    lo, hi = s.min(), s.max()
    if hi == lo:  # no spread in the data -> everyone gets the same neutral score
        return pd.Series(50.0, index=s.index)
    scaled = (s - lo) / (hi - lo) * 100
    return 100 - scaled if invert else scaled

## 1. Transit Score

Weighted blend of:
- Stop density (`Transit_Stop_Density_per_sqkm`) — **35%**
- Route density (`Transit_Route_Density_per_sqkm`) — **25%**
- Metro station count — **25%**
- Share of stops that are wheelchair-accessible (`Accessible_Stop_Count / Transit_Stop_Count`) — **15%**

In [3]:
accessible_share = (df["Accessible_Stop_Count"] / df["Transit_Stop_Count"].replace(0, np.nan)).fillna(0)

transit_components = {
    "stop_density":   (minmax(df["Transit_Stop_Density_per_sqkm"]), 0.35),
    "route_density":  (minmax(df["Transit_Route_Density_per_sqkm"]), 0.25),
    "metro_stations": (minmax(df["Metro_Station_Count"]), 0.25),
    "accessibility":  (minmax(accessible_share), 0.15),
}
df["Transit_Score_0_100"] = sum(score * w for score, w in transit_components.values())

## 2. Campus Access Score

Pure inverse distance: whichever neighborhood centroid is closest to its nearest campus (`Distance_to_Nearest_Campus_km`) scores 100; the farthest scores 0.

In [4]:
df["Campus_Access_Score_0_100"] = minmax(df["Distance_to_Nearest_Campus_km"], invert=True)

## 3. Affordability Score

- **70%** inverse-normalized average rent (mean of bachelor/1BR/2BR/3BR — cheaper = higher score)
- **30%** normalized vacancy rate (higher vacancy = easier to find a unit = more "affordable" in practice)

> Your rent data (`Rental_Data_Granularity`) is neighbourhood-level CMHC survey data (Oct 2025), so this differentiates properly across neighborhoods rather than coming out flat.

In [5]:
avg_rent = df[[
    "Average_Rent_Bachelor_CMA",
    "Average_Rent_1BR_CMA",
    "Average_Rent_2BR_CMA",
    "Average_Rent_3BR_CMA",
]].mean(axis=1)

df["Affordability_Score_0_100"] = (
    minmax(avg_rent, invert=True) * 0.70
    + minmax(df["Vacancy_Rate_CMA_pct"]) * 0.30
)

## 4. Amenity Score

Uses amenity **density** (count ÷ `Area_sq_km`), not raw counts, so a large suburb isn't penalized for needing more absolute amenities to serve the same population density.

Weights: Cafes **25%**, Groceries **25%**, Libraries **20%**, Parks **15%**, Gyms **15%**

In [6]:
area = df["Area_sq_km"].replace(0, np.nan)
amenity_weights = {
    "Cafe_Count": 0.25,
    "Grocery_Count": 0.25,
    "Library_Count": 0.20,
    "Park_Count": 0.15,
    "Gym_Count": 0.15,
}
amenity_score = 0
for col, w in amenity_weights.items():
    density = (df[col] / area).fillna(0)
    amenity_score = amenity_score + minmax(density) * w
df["Amenity_Score_0_100"] = amenity_score

## 5. Current Usability Score (overall composite)

Weights sourced from the `Recommendation_Weights` sheet:

- Affordability — **25%**
- Campus proximity — **20%**
- Transit — **20%**
- Amenity bucket (groceries + cafes + libraries + parks + gyms) — **35%**

In [7]:
df["Current_Usability_Score_0_100"] = (
    df["Affordability_Score_0_100"] * 0.25
    + df["Campus_Access_Score_0_100"] * 0.20
    + df["Transit_Score_0_100"] * 0.20
    + df["Amenity_Score_0_100"] * 0.35
)

# Round for readability
for col in [
    "Transit_Score_0_100", "Campus_Access_Score_0_100",
    "Affordability_Score_0_100", "Amenity_Score_0_100",
    "Current_Usability_Score_0_100",
]:
    df[col] = df[col].round(1)

## Results, ranked by overall usability

In [8]:
ranked = df[[
    "Neighborhood", "Transit_Score_0_100", "Campus_Access_Score_0_100",
    "Affordability_Score_0_100", "Amenity_Score_0_100",
    "Current_Usability_Score_0_100",
]].sort_values("Current_Usability_Score_0_100", ascending=False).reset_index(drop=True)
ranked

,Neighborhood,Transit_Score_0_100,Campus_Access_Score_0_100,Affordability_Score_0_100,Amenity_Score_0_100,Current_Usability_Score_0_100
0,Le Plateau-Mont-Royal,65.3,91.7,61.5,89.9,78.2
1,Côte-des-Neiges-Notre-Dame-de-Grâce,58.9,89.4,68.0,35.3,59.0
2,Outremont,41.1,94.9,57.8,44.6,57.3
3,Rosemont-La Petite-Patrie,53.4,75.2,65.2,39.3,55.8
4,Villeray-Saint-Michel-Parc-Extension,52.8,66.4,72.8,37.2,55.1
5,Montréal-Ouest,50.6,98.9,50.4,31.5,53.5
6,Westmount,49.7,90.7,53.0,33.2,52.9
7,Ville-Marie,65.6,100.0,15.0,40.6,51.1
8,Le Sud-Ouest,45.0,85.5,56.1,28.8,50.2
9,Hampstead,25.3,88.2,66.3,31.0,50.1


## Save output CSV

In [9]:
df.to_csv(OUTPUT_CSV, index=False)
print(f"Wrote {OUTPUT_CSV} with {len(df)} rows.")

Wrote Montreal_Newcomer_Navigator_Scored.csv with 34 rows.
